<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [15]:
mlflow.set_experiment(
    "assignment"
)

2026/05/22 17:45:36 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/luisg/github-classroom/ddefbcourses/atividade-04-deep-learning-i-Luis-Gustavo-Melo/notebooks/mlruns/101653067055274237', creation_time=1779482736519, experiment_id='101653067055274237', last_update_time=1779482736519, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?


**Solução**:

1. Qual o formato original das imagens?

32 pixels de altura, 32 de largura e 3 canais de cor RGB

2. Quantas features cada imagem possui após o flatten?

3072 features

3. Por que o flatten é necessário para uma MLP?

Porque as camadas densas (Fully Connected) de uma MLP aceitam apenas vetores unidimensionais (1D) como entrada. Elas não conseguem processar matrizes espaciais 3D diretamente.

4. Qual a importância da normalização para o treinamento?

Escalonar os valores de entrada para a faixa de 0 a 1 melhora a convergência do algoritmo de Gradiente Descendente, evitando gradientes explosivos e acelerando o tempo de treinamento.

In [5]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    (X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()
    
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)
    
    X_train_full = X_train_full / 255.0
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, 
        y_train_full, 
        test_size=0.2, 
        random_state=seed
    )
    
    return X_train, X_val, y_train, y_val

X_train, X_val, y_train, y_val = load_data(seed=42)
print(f"Treino: {X_train.shape}, Validação: {X_val.shape}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 648s 4us/step
Treino: (40000, 3072), Validação: (10000, 3072)


# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [6]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=50 
    )
    
    model.fit(X_train, y_train.ravel())
    
    return model

1. Quantos parâmetros existem na primeira camada?

Depende do número de neurônios ($N$) da primeira camada oculta. A fórmula é: $(3072 \text{ entradas} \times N) + N \text{ biases}$. Se a camada tiver 100 neurônios, serão $307.300$ parâmetros só na primeira camada.

2. Qual a função da ativação ReLU?

Introduzir não-linearidade à rede (permitindo aprender padrões complexos) e mitigar o problema do "desaparecimento do gradiente" (vanishing gradient) que ocorre em funções como Sigmoid.

3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

Porque as conexões são densas: cada neurônio da primeira camada oculta precisa se conectar individualmente a cada um dos milhares de pixels da imagem planificada.


# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
        "recall": recall_score(y_test, y_pred, average='macro', zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average='macro', zero_division=0)
    }
    
    df_metrics = pd.DataFrame([metrics])
    display(df_metrics)
    
    return metrics

1. O que a accuracy representa?

A taxa geral de acertos, ou seja, a proporção de imagens classificadas corretamente em relação ao total.

2. Qual a diferença entre precision e recall?

A Precision mede a exatidão dos palpites positivos (das imagens que o modelo classificou como "Gato", quantas realmente eram gatos). O Recall mede a capacidade de encontrar os positivos (de todos os gatos reais no dataset, quantos o modelo conseguiu identificar).

3. Em quais situações o f1-score é importante?

Quando o dataset está desbalanceado (muito mais exemplos de uma classe do que de outra) ou quando precisamos encontrar um meio-termo ideal entre precisão e recall, já que o F1 é a média harmônica entre os dois.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [16]:
import time
import mlflow

def run_experiment(run_name, activation, hidden_layers, learning_rate, seed=42):
    with mlflow.start_run(run_name=run_name):
        # 1. Log dos parâmetros
        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("seed", seed)
        
        start_time = time.time()
        model = train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed)
        training_time = time.time() - start_time
        
        metrics = evaluate(model, X_val, y_val)
        
        mlflow.log_metric("accuracy", metrics["accuracy"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall", metrics["recall"])
        mlflow.log_metric("f1_score", metrics["f1_score"])
        mlflow.log_metric("training_time", training_time)
        
        print(f"Experimento {run_name} finalizado em {training_time:.2f}s")
        return model, metrics


1. Qual experimento apresentou melhor desempenho?

O experimento com a arquitetura (256, 128) obteve o melhor desempenho absoluto, alcançando uma acurácia de aproximadamente 51,06% (0.5106).

2. Qual configuração apresentou maior estabilidade?

O uso do learning rate de 0.001. Fica claro ao comparar com o learning rate de 0.1, que colapsou totalmente a rede para apenas 9,7% de acurácia (basicamente o equivalente a um palpite aleatório para 10 classes).

3. Qual o benefício do rastreamento experimental?

O MLflow nos permite visualizar exatamente o custo-benefício de cada alteração. Por exemplo, pudemos cruzar os dados e perceber que pular de uma arquitetura (128, 64) para (256, 128) dobrou o tempo de treinamento (de ~80s para ~150s), mas gerou um ganho de acurácia de apenas 1,2%. Fazer esse tipo de análise manualmente seria inviável.


# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [17]:
activations = ['logistic', 'tanh', 'relu']

for act in activations:
    print(f"\nTreinando com ativação: {act}")
    run_experiment(
        run_name=f"activation_{act}",
        activation=act,
        hidden_layers=(64,),
        learning_rate=0.001 
    )


Treinando com ativação: logistic


,accuracy,precision,recall,f1_score
0,0.48,0.482048,0.480687,0.477042


Experimento activation_logistic finalizado em 59.01s

Treinando com ativação: tanh


,accuracy,precision,recall,f1_score
0,0.4361,0.439036,0.438612,0.425257


Experimento activation_tanh finalizado em 50.28s

Treinando com ativação: relu


,accuracy,precision,recall,f1_score
0,0.4495,0.444493,0.451086,0.437888


Experimento activation_relu finalizado em 64.22s


1. Qual ativação apresentou melhor convergência?

No nosso recorte específico (utilizando a rede de tamanho (64,)), a ativação logistic obteve a melhor acurácia final (48%), enquanto a relu convergiu para 44,9%.

2. Qual ativação apresentou maior estabilidade?

A logistic e a relu mantiveram resultados competitivos entre si. A ativação tanh teve o desempenho ligeiramente mais baixo do grupo (43,6%).

3. Houve diferenças significativas no treinamento?

Sim, embora a variação de tempo tenha sido pequena (entre 50 e 64 segundos para as três), a variação de acurácia mostrou que a escolha da função afeta diretamente a capacidade da rede de extrair padrões dos pixels.

4. Por que a ReLU é amplamente utilizada em Deep Learning?

Apesar da logistic ter ido bem nessa rede rasa, a ReLU é o padrão da indústria porque, em arquiteturas profundas (com várias camadas), funções como a logistic sofrem com o "desaparecimento do gradiente", onde o erro se dilui até chegar a zero antes de atualizar as primeiras camadas. O gradiente constante da ReLU evita isso e o seu cálculo computacional (apenas max(0, x)) é muito mais veloz.


# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [18]:
architectures = [(32,), (64,), (128, 64), (256, 128)]

for arch in architectures:
    print(f"\nTreinando com arquitetura: {arch}")
    run_experiment(
        run_name=f"arch_{arch}",
        activation='relu',
        hidden_layers=arch,
        learning_rate=0.001
    )


Treinando com arquitetura: (32,)


,accuracy,precision,recall,f1_score
0,0.3681,0.367752,0.367986,0.355298


Experimento arch_(32,) finalizado em 34.44s

Treinando com arquitetura: (64,)


,accuracy,precision,recall,f1_score
0,0.4495,0.444493,0.451086,0.437888


Experimento arch_(64,) finalizado em 61.78s

Treinando com arquitetura: (128, 64)


,accuracy,precision,recall,f1_score
0,0.4985,0.504616,0.498785,0.495582


Experimento arch_(128, 64) finalizado em 79.96s

Treinando com arquitetura: (256, 128)


,accuracy,precision,recall,f1_score
0,0.5106,0.514656,0.511537,0.508748


Experimento arch_(256, 128) finalizado em 149.95s


1. Redes maiores sempre melhoraram os resultados?

Neste experimento sim, houve uma melhora contínua: saímos de 36,8% com a rede de (32,) neurônios e chegamos ao topo de 51,06% com a rede (256, 128).

2. Qual arquitetura apresentou melhor tradeoff?

A arquitetura (128, 64). Ela atingiu uma acurácia excelente de 49,85% em apenas 79,9 segundos. Para ganhar pouco mais de 1% extra, a rede (256, 128) precisou de quase o dobro do tempo (149,9 segundos).

3. Houve sinais de overfitting?

O rendimento decrescente da rede (256, 128) é um indício claro. Aumentar excessivamente a quantidade de parâmetros fez o tempo de treinamento explodir sem um ganho proporcional na validação, sugerindo que a rede pode estar começando a decorar os dados de treino em vez de aprender regras generalizáveis.

4. Qual arquitetura apresentou maior custo computacional?

A arquitetura (256, 128), que exigiu 149,95 segundos para finalizar o treinamento.


# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [19]:
learning_rates = [0.1, 0.01, 0.001]

for lr in learning_rates:
    print(f"\nTreinando com learning rate: {lr}")
    run_experiment(
        run_name=f"lr_{lr}",
        activation='relu',
        hidden_layers=(128, 64),
        learning_rate=lr
    )


Treinando com learning rate: 0.1


,accuracy,precision,recall,f1_score
0,0.0973,0.00973,0.1,0.017734


Experimento lr_0.1 finalizado em 120.82s

Treinando com learning rate: 0.01


,accuracy,precision,recall,f1_score
0,0.3995,0.403991,0.399847,0.391477


Experimento lr_0.01 finalizado em 138.24s

Treinando com learning rate: 0.001


,accuracy,precision,recall,f1_score
0,0.4985,0.504616,0.498785,0.495582


Experimento lr_0.001 finalizado em 136.65s


1. Qual learning rate apresentou melhor desempenho?

O learning rate de 0.001, que estabilizou e alcançou 49,85% de acurácia.

2. Qual apresentou maior instabilidade?

O learning rate extremo de 0.1. O modelo divergiu completamente, travando em míseros 9,7% de acurácia e o f1-score desabou para 0.017.

3. O que acontece quando o learning rate é muito alto?

A taxa de atualização dos pesos fica tão agressiva que o algoritmo "salta" por cima do ponto ideal da função de perda (mínimo global). Os erros passam a oscilar drasticamente e a rede destrói qualquer padrão que havia começado a aprender (como evidenciado pelo teste do 0.1).

4. O que acontece quando o learning rate é muito baixo?

Os passos de ajuste de peso se tornam minúsculos. O treinamento levaria uma eternidade em número de épocas para atingir uma boa precisão, e o modelo corre o sério risco de ficar preso em um "mínimo local" (um ponto ruim de erro) porque seus saltos não têm força suficiente para tirá-lo de lá.


# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

A análise dos nossos logs no MLflow demonstra claramente como hiperparâmetros determinam o sucesso ou o fracasso de uma MLP no dataset CIFAR-10. O learning rate provou ser o elemento mais sensível da arquitetura: ajustes para valores altos (0.1) resultaram em divergência catastrófica (9,7% de acurácia). Quanto ao tamanho da rede, observamos um clássico dilema de custo-benefício: expandir as camadas para (256, 128) cravou o recorde absoluto de acurácia (51%), mas elevou o tempo computacional para 150 segundos, comprovando que a versão (128, 64) ofereceu o melhor equilíbrio geral. Apesar da função logistic ter se saído marginalmente melhor que a ReLU nas redes rasas do nosso teste, sabemos que para contornar limitações e construir redes profundas verdadeiras, a não-linearidade da ReLU é mandatória.

1. Qual configuração apresentou melhor resultado final?

A rede com a arquitetura maior (256, 128), alcançando 51,06% de acurácia.

2. Quais foram as principais dificuldades observadas?

Equilibrar o altíssimo custo de tempo da CPU em arquiteturas maiores (que chegaram a demorar quase 3 minutos) e descobrir a faixa exata de learning rate que não arruinasse a convergência.

3. Por que MLPs possuem limitações para imagens?

O uso obrigatório do flatten destrói a informação espacial. Os pixels que representam o focinho de um cachorro, que são vizinhos na matriz 2D original, perdem esse contexto de vizinhança no vetor 1D. Além disso, as MLPs tratam os mesmos objetos em locais diferentes da tela como atributos inteiramente novos, pois carecem de invariância à translação.

4. Como o backpropagation contribui para o aprendizado da rede?

Ele funciona como o "crítico" do modelo. Após a rede arriscar uma previsão, o backpropagation utiliza as derivadas para propagar o tamanho do erro da última camada de volta para a primeira. Ele diz ao otimizador exatamente qual peso deve ser reduzido ou aumentado para que o erro seja minimizado na próxima rodada, possibilitando que o Gradiente Descendente atue na direção correta.
